# Moon-SoC Hierarchical Macro Synthesis & Top-Level Assembly in LibreLane

This notebook synthesizes and hardens **Moon-SoC** hierarchically into independent physical macros:
- **System Control (AON/CRG/PMU)** (`moon_sysctrl_subsystem`)
- **Peripheral Subsystem** (`moon_periph_subsystem`)
- **System Fabric** (`moon_sys_fabric`)
- **Memory Subsystem** (`moon_mem_subsystem`)
- **Cache Tile** (`moon_cache_tile`)
- **CPU Compute Tile** (`moon_cpu_tile`)

And then assembles the complete top-level SoC (`moon_soc_top`) with the I/O & Memory PHY boundary cells placed at the pad frame.

## 1. Setup Nix and LibreLane

In [ ]:
# @title Setup Nix & Install LibreLane
import os, sys, shutil, subprocess, tempfile

os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        with tempfile.TemporaryDirectory() as d:
            installer_path = os.path.join(d, "nix")
            !curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix > {installer_path}
            subprocess.run([
                "bash", installer_path, "install",
                "--prefer-upstream-nix", "--no-confirm",
                "--extra-conf",
                "extra-substituters = https://nix-cache.fossi-foundation.org\nextra-trusted-public-keys = nix-cache.fossi-foundation.org:3+K59iFwXqKsL7BNu6Guy0v+uTlwsxYQxjspXzqLYQs=\n"
            ], check=True)

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"

!pip install -q librelane ciel pyyaml
print("Nix & LibreLane setup complete!")

## 2. Download and Enable Sky130A PDK

In [ ]:
import ciel
from ciel.source import StaticWebDataSource
import os, yaml

pdk_root = os.path.expanduser("~/.ciel")
pdk = "sky130"

ciel.enable(
    ciel.get_ciel_home(pdk_root),
    pdk,
    data_source=StaticWebDataSource("https://fossi-foundation.github.io/ciel-releases"),
)
print("Sky130 PDK enabled successfully!")

## 3. Clone / Locate Moon-SoC Repository

In [ ]:
repo_path = "/content/moon-soc" if "google.colab" in sys.modules else os.getcwd()
if not os.path.exists(repo_path) and "google.colab" in sys.modules:
    !git clone https://github.com/phoshp/moon-soc.git {repo_path}

synth_dir = os.path.join(repo_path, "synth")
print(f"Using synthesis directory: {synth_dir}")

## 4. Harden Macro Partitions Separately

We harden each functional subsystem individually. Each block produces `.lef`, `.lib`, and `.gds` files in its `runs/latest/final/` directory.

In [ ]:
from librelane.flows import Flow

macros = [
    ("System Control (AON/CRG/PMU)", "synth/macros/sysctrl_subsystem"),
    ("Peripheral Subsystem",         "synth/macros/periph_subsystem"),
    ("System Fabric",                "synth/macros/sys_fabric"),
    ("Memory Subsystem",             "synth/macros/mem_subsystem"),
    ("Cache Tile",                   "synth/macros/cache_tile"),
    ("CPU Compute Tile",             "synth/macros/cpu_tile")
]

for label, rel_dir in macros:
    config_dir = os.path.join(repo_path, rel_dir)
    print(f"\n{'='*70}\n>>> Hardening Macro: {label} ({rel_dir})\n{'='*70}")
    flow = Flow.factory.get("Classic")(config_dir=config_dir)
    flow.start()
    print(f"[DONE] Macro {label} hardened successfully!")

## 5. Assemble Top-Level Moon-SoC Chip

Now synthesize and route the top-level `moon_soc_top`. Yosys reads the blackbox macro stubs and elaborates the inter-macro connections in **seconds**! OpenROAD then places the hardened macro LEFs according to `macro_placement.cfg` and routes the top-level chip.

In [ ]:
top_config_dir = os.path.join(synth_dir, "top")
print(f"Assembling Top-Level SoC with Macro Placement from: {top_config_dir}")

flow_top = Flow.factory.get("Classic")(config_dir=top_config_dir)
flow_top.start()
print("\n[SUCCESS] Moon-SoC Top-Level Chip implementation completed!")

## 6. Inspect Final Layout (GDSII / KLayout)

In [ ]:
top_run = os.path.join(top_config_dir, "runs", "latest", "final", "gds", "moon_soc_top.gds")
if os.path.exists(top_run):
    print(f"Top-level GDSII available at: {top_run}")
    print(f"Size: {os.path.getsize(top_run) / (1024*1024):.2f} MB")
else:
    print("Run output will be placed in runs/latest/final/gds/")